## Organized and Finalized Training Script ##

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets
import os
import kagglehub

In [65]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

Path to competition files: C:\Users\Admin\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques


### Load Data ###

In [66]:
df = pd.read_csv(os.path.join(path, 'train.csv'))

gr_liv_area = df["GrLivArea"]
garage_area = df["GarageArea"]
year_built = df["YearBuilt"]
year_remodelled = df["YearRemodAdd"]
year_sold = df["YrSold"]
overall_qual = df["OverallQual"]
overall_cond = df["OverallCond"]
neighborhood = df["Neighborhood"]
y_df = df["SalePrice"]

### Build Features ###

In [67]:
remodelled_age = year_sold - year_remodelled
built_age = year_sold - year_built
# YearRemodAdd seems to be floored at 1950 so flag those rows
remod_floor_flag = np.zeros(remodelled_age.size)
remod_floor_flag[year_remodelled == 1950] = 1

In [68]:
# convert to one hot before splitting training and validation to avoid missing any neighborhoods in either set
neighborhood_names = np.unique(neighborhood)
# Dropped NAmes (baseline neighborhood)
neighborhood_names = neighborhood_names[neighborhood_names != "NAmes"]
neighborhood_one_hot = {}
for name in neighborhood_names:
    neighborhood_one_hot[name] = (neighborhood == name).astype(float)

X_arr = np.column_stack((gr_liv_area, garage_area, overall_qual, built_age, remodelled_age, remod_floor_flag))
for n in neighborhood_one_hot:
    X_arr = np.column_stack((X_arr, neighborhood_one_hot[n]))

In [69]:
# turn y into log_y so that errors are converted from absolute to relative
log_y_arr = np.log(y_df).to_numpy()

### Split Training and Validation ###

In [70]:
rng = np.random.default_rng(seed=912)
shuffled_indices = rng.permutation(X_arr.shape[0])

validation_cutoff = X_arr.shape[0] - int(X_arr.shape[0] / 5)
train_indices = shuffled_indices[0:validation_cutoff]
valid_indices = shuffled_indices[validation_cutoff:]

X_train = X_arr[train_indices]
X_valid = X_arr[valid_indices]

log_y_train = log_y_arr[train_indices]
log_y_valid = log_y_arr[valid_indices]

### Normalize Input Features ###

In [71]:
def zscore_normalize_features(X: np.ndarray) -> np.ndarray:
    mean = np.mean(X, axis = 0)
    std = np.std(X, axis = 0)
    X_normal = (X - mean) / std
    return X_normal

In [72]:
norm_X_train = zscore_normalize_features(X_train)
norm_X_valid = zscore_normalize_features(X_valid)

### Train ###

In [73]:
def cost(X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float) -> float:
    f_wb = np.sum(X * w, axis = 1) + b
    diff = (f_wb - y) ** 2
    sum = np.sum(diff)
    cost = sum / (2 * X.shape[0])
    return cost

In [74]:
def compute_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray, b: float) -> tuple:
    f_wb = np.sum((X * w), axis = 1) + b
    diff = (f_wb - y)
    diff = diff.reshape(diff.shape[0],1)
    mult = X * diff
    sum = np.sum(mult, axis = 0)
    gradient_w = sum / (X.shape[0])
    gradient_b = np.sum(diff) / (X.shape[0])
    return gradient_w, gradient_b

In [78]:
def train(X: np.ndarray, y: np.ndarray, alpha, cost, compute_gradient):
    # initialize an array for weights
    w = np.zeros(X.shape[1])
    b = 0
    iterations = 0
    prev_cost = 0
    curr_cost = cost(X, y, w, b)

    while (abs(curr_cost - prev_cost) > 0.000000000000001) and iterations < 100000:
        prev_cost = curr_cost
        delta_w, delta_b = compute_gradient(X, y, w, b)
        if iterations == 0:
            print(delta_w)
        w = w - alpha * delta_w
        b = b - alpha * delta_b
        curr_cost = cost(X, y, w, b)
        if iterations % 50 == 0:
            print(f"cost: {curr_cost}")
        iterations += 1

    return w, b, iterations

In [79]:
w, b, iterations = train(norm_X_train, log_y_train, 0.1, cost, compute_gradient)
print(iterations)

[-0.27932642 -0.25734503 -0.32742066  0.2338837   0.23317397  0.15626904
 -0.0150517   0.00872757  0.04431621  0.07556531 -0.03006479 -0.04721539
 -0.04322553  0.08076312 -0.02876812  0.10232335  0.06722349  0.01968635
  0.0130855  -0.01891284 -0.11936625 -0.13960441  0.08529054  0.03042505
  0.04839915 -0.01078311 -0.06921251 -0.07592298 -0.05414625 -0.02651252]
cost: 58.616102451568125
cost: 0.013948866595222436
cost: 0.012375142000046541
cost: 0.012373307524992634
cost: 0.01237298123926337
cost: 0.012372904020409118
cost: 0.012372882960280145
cost: 0.012372876686838894
cost: 0.012372874696042587
cost: 0.012372874031763816
cost: 0.012372873800841776
cost: 0.01237287371789597
cost: 0.012372873687343737
cost: 0.012372873675879248
cost: 0.012372873671519832
cost: 0.012372873669846754
cost: 0.012372873669200582
cost: 0.012372873668949947
cost: 0.012372873668852454
911
